# IQM Superconducting Exploration

Route a circuit by hand for a nearest-neighbor lattice and measure the SWAP tax.

**Objectives:**
- Understand why a superconducting (lattice) device cannot run a two-qubit gate between non-adjacent qubits
- Build a logical circuit with a CNOT across a linear chain, then route it by inserting SWAP gates
- Prove the routed circuit computes the *same* state as the logical one
- Measure the SWAP tax: count two-qubit gates and depth before vs. after routing
- See how the tax grows with the distance between control and target

**Reference:** See [`../GUIDE.md`](../GUIDE.md) for concept explanations and context.
<!-- browser-runnable -->

## 0. The wiring constraint

IonQ's trapped ions are **all-to-all**: any qubit can entangle with any other directly. IQM's
superconducting transmons are not. They sit on a fixed **lattice**, and a two-qubit gate can only
run between two qubits that are physically **adjacent**.

We model the simplest lattice — a **linear chain** `0-1-2-3-4`. The only legal two-qubit gates are
between neighbors: `|i - j| == 1`. To apply a CNOT between distant qubits (say 0 and 4), the
compiler must first **SWAP** states along the chain to bring them together. Every SWAP is itself a
two-qubit gate, so it adds depth — and, on a real NISQ device, more noise. That is the **SWAP tax**.

In this notebook we build a logical circuit, route it by hand for the chain, and prove the routed
version is correct while paying strictly more in two-qubit gates and depth.

In [ ]:
# Setup: run this cell first.

from braket.circuits import Circuit
from braket.devices import LocalSimulator
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(0)            # deterministic sampling for the asserts below
device = LocalSimulator()

CHAIN = 5                    # linear chain of qubits: 0-1-2-3-4 (nearest-neighbor only)


def two_qubit_gate_count(circ):
    """Count two-qubit gates (CNOT, SWAP, CZ) in a circuit."""
    return sum(1 for ins in circ.instructions if ins.operator in ("CNOT", "SWAP", "CZ"))


def span(circ, n):
    """Touch the top qubit so state_vector() spans all n qubits (big-endian, qubit 0 = MSB)."""
    circ.i(n - 1)
    return circ


print("Linear chain of", CHAIN, "qubits: 0-1-2-3-4")
print("Legal two-qubit gates only between adjacent qubits where |i - j| == 1.")

## 1. The logical circuit

First, the circuit we *want*: a Hadamard on qubit 0 to create a superposition, then a CNOT from
qubit 0 to qubit 4. On an all-to-all machine this is exactly two gates and produces a Bell-like
pair stretched across the chain ends:
$\;\tfrac{1}{\sqrt{2}}\big(\ket{00000} + \ket{10001}\big)$.

We put the Hadamard *first* so the entanglement is observable in the measured counts later. We call
`span(..., CHAIN)` so the state vector covers all five qubits.

In [ ]:
logical = Circuit()
logical.h(0)          # superposition on qubit 0 so the CNOT entangles, not just copies
logical.cnot(0, 4)    # logical CNOT across the whole chain (qubits 0 and 4 are NOT adjacent)
span(logical, CHAIN)

print(logical)
print()
print("logical two-qubit gates:", two_qubit_gate_count(logical))
print("logical depth          :", logical.depth)

## 2. A router for the linear chain

The gate `cnot(0, 4)` is illegal on the chain because `|0 - 4| = 4 > 1`. Here is the standard fix.

To apply `cnot(control, target)` when they are non-adjacent:
1. **SWAP** the control one step at a time along the shortest path until it sits *next to* the target.
2. Apply the (now legal, adjacent) **CNOT**.
3. **SWAP** back in reverse to restore the original qubit labeling.

Because `SWAP` is its own inverse, the swaps in step 3 exactly undo step 1: the only logically-active
operation that survives is `cnot(control, target)` on the original wiring. So the routed circuit
implements the *same unitary* — we verify that in the next section.

In [ ]:
def route_cnot_linear(circ, control, target):
    """Append a CNOT(control, target) to `circ`, legal for a linear nearest-neighbor chain.

    Adjacent qubits: emit the CNOT directly. Otherwise SWAP the control along the
    shortest path to the target, apply the CNOT, then SWAP back (restoring labels).
    """
    if control == target:
        raise ValueError("control and target must differ")
    if abs(control - target) == 1:
        circ.cnot(control, target)
        return circ

    step = 1 if target > control else -1
    pos = control
    # 1. walk the control toward the target until it is adjacent
    while abs(pos - target) > 1:
        circ.swap(pos, pos + step)
        pos += step
    # 2. apply the now-adjacent CNOT
    circ.cnot(pos, target)
    # 3. undo the swaps to restore the original qubit labeling
    while pos != control:
        circ.swap(pos, pos - step)
        pos -= step
    return circ


routed = Circuit()
routed.h(0)
route_cnot_linear(routed, 0, 4)
span(routed, CHAIN)

print(routed)
print()
print("routed two-qubit gates:", two_qubit_gate_count(routed))
print("routed depth          :", routed.depth)

## 3. Routing is correct

The whole point of routing is that it must not change *what the circuit computes* — only *how* it
runs on constrained hardware. We compare the two state vectors directly. `state_vector()` returns
the full amplitude array (big-endian, qubit 0 = leftmost bit), and both circuits span all five
qubits, so this is an exact, sampling-free check.

In [ ]:
sv_logical = logical.state_vector()
sv_routed = routed.state_vector()

assert sv_logical.shape == sv_routed.shape == (2 ** CHAIN,)
assert np.allclose(sv_logical, sv_routed), "routed circuit computes a different state!"
print("state_vector match:", np.allclose(sv_logical, sv_routed))

print("\nNon-zero amplitudes of the shared state (big-endian kets):")
for idx in np.flatnonzero(np.abs(sv_logical) > 1e-9):
    print(f"  |{idx:0{CHAIN}b}>  amplitude = {sv_logical[idx]:.4f}")
print("\nThis is the Bell-like pair (|00000> + |10001>)/sqrt(2): qubits 0 and 4 entangled.")

## 4. The SWAP tax

Routing preserved the result — but it was not free. The logical CNOT(0, 4) crosses a distance of 4,
so the router inserts $2\times(4-1) = 6$ SWAP gates (three out, three back) around the single CNOT.
That is **7 two-qubit gates** where the logical circuit had **1**, and the serial SWAP chain
lengthens the critical path, so depth can only grow.

We assert the tax exactly: strictly more two-qubit gates, and depth no smaller than the logical
circuit's.

In [ ]:
n_log = two_qubit_gate_count(logical)
n_rt = two_qubit_gate_count(routed)

assert n_rt > n_log, "routing must add two-qubit gates (the SWAP tax)"
assert routed.depth >= logical.depth, "routing cannot reduce depth"

print(f"two-qubit gates : {n_log} -> {n_rt}  (+{n_rt - n_log} SWAPs)")
print(f"depth           : {logical.depth} -> {routed.depth}")
print("SWAP tax confirmed: strictly more two-qubit gates, and depth did not shrink.")

## 5. The entanglement survives routing

A final sanity check on real measurement statistics. The shared state correlates qubit 0 with
qubit 4 perfectly, so every sampled bitstring must have matching first and last bits — only
`00000` and `10001` can appear. The sampling is seeded, so this is reproducible in CI.

In [ ]:
counts = device.run(routed, shots=2000).result().measurement_counts
print("measurement counts:", dict(counts))

for bitstring in counts:
    assert bitstring[0] == bitstring[4], f"q0 and q4 disagree: {bitstring}"
print("q0 == q4 in every shot: the entanglement created by the logical CNOT survived routing.")

## 6. The tax grows with distance

The deeper the two qubits are apart on the chain, the more SWAPs the router must insert: a CNOT
spanning distance $k$ costs $2(k-1)$ SWAPs plus the CNOT, i.e. $2k-1$ two-qubit gates. We sweep
`cnot(0, k)` for `k = 1..4`, verify each routed circuit is still correct, and chart the tax. This is
exactly why a circuit that looks shallow on paper can balloon in depth on a lattice device — and why
all-to-all machines (IonQ) are prized for densely connected problems.

In [ ]:
distances = [1, 2, 3, 4]
log_2q, rt_2q, rt_depth = [], [], []
for k in distances:
    L = Circuit(); L.h(0); L.cnot(0, k); span(L, CHAIN)
    R = Circuit(); R.h(0); route_cnot_linear(R, 0, k); span(R, CHAIN)
    assert np.allclose(L.state_vector(), R.state_vector()), f"routing wrong for cnot(0,{k})"
    log_2q.append(two_qubit_gate_count(L))
    rt_2q.append(two_qubit_gate_count(R))
    rt_depth.append(R.depth)

print(f"{'dist k':>6} {'logical 2q':>12} {'routed 2q':>10} {'routed depth':>13}")
for k, a, b, d in zip(distances, log_2q, rt_2q, rt_depth):
    print(f"{k:>6} {a:>12} {b:>10} {d:>13}")

fig, ax = plt.subplots(figsize=(7, 4))
x = np.arange(len(distances)); w = 0.38
ax.bar(x - w / 2, log_2q, w, label="logical (all-to-all)", color="#232f3e")
ax.bar(x + w / 2, rt_2q, w, label="routed (linear chain)", color="#ff9900")
ax.set_xticks(x); ax.set_xticklabels([f"dist {k}" for k in distances])
ax.set_ylabel("two-qubit gate count")
ax.set_title("The SWAP tax grows with qubit distance on a lattice")
ax.legend()
plt.tight_layout()
plt.show()

## Exercises

In [ ]:
# Exercise 1: Route a CNOT that does NOT start at qubit 0.
# Build the logical circuit h(1) + cnot(1, 4) on the chain, route it with
# route_cnot_linear, and assert the state vectors match. How many SWAPs did it take?

# TODO: Your code here
# logical_b = Circuit(); logical_b.h(1); logical_b.cnot(1, 4); span(logical_b, CHAIN)
# routed_b = Circuit(); routed_b.h(1); route_cnot_linear(routed_b, 1, 4); span(routed_b, CHAIN)
# assert np.allclose(logical_b.state_vector(), routed_b.state_vector())
# print("SWAPs added:", two_qubit_gate_count(routed_b) - two_qubit_gate_count(logical_b))

In [ ]:
# Exercise 2: A cheaper alternative. Instead of swapping back, you can LEAVE the
# control swapped and relabel which physical qubit now holds the logical control.
# Write route_cnot_leave(circ, control, target) that omits step 3, returns the
# physical index where the control ended up, and uses HALF the SWAPs.
# (Hint: the final state vector will differ because the labels are permuted -- that
# is the trade-off. Track the mapping yourself if you keep routing more gates.)

# TODO: Your code here
# def route_cnot_leave(circ, control, target):
#     step = 1 if target > control else -1
#     pos = control
#     while abs(pos - target) > 1:
#         circ.swap(pos, pos + step); pos += step
#     circ.cnot(pos, target)
#     return pos  # the control now lives here
#
# c = Circuit(); c.h(0); final_pos = route_cnot_leave(c, 0, 4); span(c, CHAIN)
# print("control ended at physical qubit", final_pos,
#       "with", two_qubit_gate_count(c), "two-qubit gates")

## Summary

- A nearest-neighbor lattice (IQM superconducting) can only run two-qubit gates between **adjacent**
  qubits; a CNOT between distant qubits must be **routed** with SWAP gates.
- A simple linear-chain router swaps the control to the target, applies the CNOT, then swaps back.
  Because SWAP is self-inverse, the routed circuit computes the **identical state** — we proved it
  with `np.allclose` on the full state vectors.
- Correctness is not free: routing pays the **SWAP tax** — strictly more two-qubit gates and no less
  depth. A CNOT across distance $k$ costs $2k-1$ two-qubit gates instead of 1.
- More two-qubit gates means more depth, and on a real NISQ device more depth means more noise — the
  reason all-to-all machines (IonQ) shine on densely connected problems and lattice machines (IQM)
  shine on circuits with local structure.

**Next:** [`04-quera-analog.ipynb`](04-quera-analog.ipynb) — neutral-atom analog computation, where you
place atoms in a geometry and evolve them under a Hamiltonian instead of routing gates at all.